# 实验十 K 近邻

K 近邻算法根据样本之间的距离进行分类。对于一个待预测样本，选择距离最近的 K 个训练样本，并根据这些近邻样本的类别投票确定预测结果。

本实验先使用二维样本理解 KNN 的距离计算和投票过程，再使用鸢尾花数据完成分类，重点调整 `n_neighbors`、`weights` 和 `p`，观察测试准确率变化。

## 1. 使用二维样本完成基础 KNN 分类

计算待预测点与训练样本的距离，选择最近的 K 个样本并输出预测类别。

In [ ]:
import numpy as np
from math import sqrt
from collections import Counter

X_train = np.array([[1.4, 0.2],
                    [1.7, 0.4],
                    [1.5, 0.1],
                    [4.7, 1.4],
                    [4.9, 1.5],
                    [4.0, 1.3]])
y_train = np.array([0, 0, 0, 1, 1, 1])
test_point = np.array([4.2, 1.5])

# 计算待预测点与所有训练样本的距离
distances = [sqrt(np.sum((row - test_point) ** 2)) for row in X_train]
print('全部距离：', distances)

# 按距离从小到大排序，得到索引
nearest = np.argsort(distances)

k = 3
top_k_y = [y_train[i] for i in nearest[:k]]
print('最近 k 个样本的类别：', top_k_y)

predict = Counter(top_k_y).most_common(1)
print('预测类别：', predict)

> **操作要求**：将 k 修改为 1、3、5，记录待预测点的类别，观察近邻数量变化是否会改变投票结果。

## 2. 修改近邻数 n_neighbors

使用鸢尾花数据，将 `n_neighbors` 分别设置为 1、3、5、7、9，记录测试准确率并选择较合适的 K。

In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

iris = datasets.load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = (
    train_test_split(X, y, test_size=0.2, random_state=666, stratify=y))

for k in [1, 3, 5, 7, 9]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    print(k, model.score(X_test, y_test))

> **操作要求**：记录不同 K 值下的测试准确率，分析 K 值过小和过大时模型结果的变化。

## 3. 修改距离权重 weights

比较 `weights="uniform"` 与 `weights="distance"`，观察距离加权是否改善分类效果。

In [ ]:
for method in ["uniform", "distance"]:
    for k in [3, 5, 7]:
        model = KNeighborsClassifier(n_neighbors=k, weights=method)
        model.fit(X_train, y_train)
        print('weights =', method, 'k =', k, '准确率 =', round(model.score(X_test, y_test), 4))

# 搜索较优的 K 和权重方式
best_score, best_k = 0.0, -1
best_method = ''
for method in ['uniform', 'distance']:
    for k in range(1, 11):
        model = KNeighborsClassifier(n_neighbors=k, weights=method)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        if score > best_score:
            best_score, best_k, best_method = score, k, method

print('最佳得分：', best_score, '最佳 k：', best_k, '最佳权重：', best_method)

> **操作要求**：比较两种权重方式。当 `weights="distance"` 时，距离较近的样本对预测结果影响更大。

## 4. 修改距离参数 p

将 `p` 设置为 1、2、3，比较曼哈顿距离、欧氏距离及其他闵可夫斯基距离的结果。

In [ ]:
for p_value in [1, 2, 3]:
    model = KNeighborsClassifier(n_neighbors=5, weights="distance", p=p_value)
    model.fit(X_train, y_train)
    print('p =', p_value, '准确率 =', round(model.score(X_test, y_test), 4))

> **操作要求**：比较 p=1、2、3 的结果，判断不同距离计算方式是否会影响近邻选择和分类准确率。

## 5. 使用 GridSearchCV 搜索参数

使用 `GridSearchCV` 同时搜索 `n_neighbors`、`weights` 和 `p`，输出最佳参数及最佳得分。

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"weights": ["uniform"],
     "n_neighbors": range(1, 11)},
    {"weights": ["distance"],
     "n_neighbors": range(1, 11),
     "p": [1, 2, 3]}
]

grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid.fit(X_train, y_train)
print('最佳参数：', grid.best_params_)
print('最佳得分：', grid.best_score_)

> **操作要求**：记录网格搜索得到的最佳参数，并与前面单独调节参数时的结果进行比较。

## 6. 创新拓展实践：标准化对 KNN 的影响

使用乳腺癌数据集比较"未标准化数据"和"标准化数据"的 KNN 分类结果。

任务：分别使用 X 和 X_std 划分训练集、训练模型并计算测试准确率，分析不同特征量纲对距离计算和 KNN 分类结果的影响。

> 注意：标准化参数应只在训练数据上拟合；拓展实验中可使用 `Pipeline` 避免测试数据参与标准化。

In [ ]:
from sklearn.preprocessing import StandardScaler

cancer = datasets.load_breast_cancer()
X, y = cancer.data, cancer.target

# 未标准化
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
model_raw = KNeighborsClassifier(n_neighbors=5)
model_raw.fit(X_train_raw, y_train_raw)
print('未标准化测试准确率：', round(model_raw.score(X_test_raw, y_test_raw), 4))

# 标准化（只在训练数据上拟合标准化参数）
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_raw)
X_test_std = scaler.transform(X_test_raw)
model_std = KNeighborsClassifier(n_neighbors=5)
model_std.fit(X_train_std, y_train_raw)
print('标准化测试准确率：', round(model_std.score(X_test_std, y_test_raw), 4))